# AstroCLIMB: improved memory-safe Kaggle pipeline

This notebook extends the starter with five practical improvements:

1. CLIP embeddings are cached persistently by SHA-256 object hash, never by the full Base64 value.
2. Image–image pairs receive perceptual-hash similarity features.
3. Caption–caption pairs receive word- and character-level TF-IDF cosine similarities.
4. Validation groups connected/repeated objects by hash to reduce leakage.
5. Out-of-fold probabilities tune class-specific decision thresholds for macro-F1, and a nonlinear histogram gradient-boosting classifier is compared with logistic regression.

The raw CSV files are still streamed in tiny chunks. Feature shards, the SQLite embedding cache, TF-IDF models, and trained classifiers are resumable artifacts under `/kaggle/working/astroclimb_improved`. Use a Kaggle GPU for CLIP extraction.

In [ ]:
# Usually available on Kaggle. Uncomment only if an import fails.
# !pip install -q transformers sentencepiece

In [ ]:
import base64, gc, hashlib, io, json, os, pickle, sqlite3, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from scipy.fft import dctn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore', message='.*DecompressionBomb.*')

LABELS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
WORK = Path('/kaggle/working/astroclimb_improved')
WORK.mkdir(parents=True, exist_ok=True)
CSV_CHUNK_SIZE = 8
MODEL_BATCH_SIZE = 16
MAX_TEXT_LENGTH = 77
SEED = 42
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Locate files and load CLIP

In [ ]:
def find_file(filename):
    matches = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under /kaggle/input')
    print(filename, 'matches:', [str(p) for p in matches])
    return matches[0]

TRAIN_CSV = find_file('train.csv')
TEST_CSV = find_file('test.csv')
SAMPLE_CSV = find_file('sample_submission.csv')

MODEL_NAME = 'openai/clip-vit-base-patch32'  # or a local attached model directory
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).eval().to(DEVICE)
if DEVICE == 'cuda':
    model = model.half()
EMBED_DIM = int(model.config.projection_dim)
print('CLIP dimension:', EMBED_DIM)

## Hashing and persistent embedding cache

Each object is represented by a fixed-size SHA-256 digest. The SQLite cache maps that digest to a CLIP vector, so repeated captions or figures are encoded once across train, test, restarts, and notebook sessions. The large Base64 payload is used only to calculate its digest and decode a cache miss; it is never retained as a dictionary key.

In [ ]:
def is_png_b64(value):
    return isinstance(value, str) and value.lstrip().startswith('iVBOR')

def object_hash(value):
    value = value if isinstance(value, str) else ''
    modality = b'I\0' if is_png_b64(value) else b'T\0'
    return hashlib.sha256(modality + value.encode('utf-8')).hexdigest()

def decode_png(value):
    raw = base64.b64decode(value.strip(), validate=False)
    with Image.open(io.BytesIO(raw)) as im:
        im.thumbnail((2048, 2048))
        return im.convert('RGB').copy()

def l2_normalize(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-8, None)

CACHE_DB = WORK / 'object_cache.sqlite'
cache = sqlite3.connect(CACHE_DB)
cache.execute('PRAGMA journal_mode=WAL')
cache.execute('CREATE TABLE IF NOT EXISTS embeddings (hash TEXT PRIMARY KEY, vector BLOB NOT NULL)')
cache.commit()

def cache_get_many(hashes):
    found = {}
    # SQLite commonly limits one query to 999 parameters.
    for start in range(0, len(hashes), 900):
        batch = list(dict.fromkeys(hashes[start:start + 900]))
        if not batch:
            continue
        marks = ','.join('?' for _ in batch)
        rows = cache.execute(f'SELECT hash, vector FROM embeddings WHERE hash IN ({marks})', batch)
        found.update({h: np.frombuffer(blob, dtype=np.float16).astype(np.float32) for h, blob in rows})
    return found

def cache_put_many(items):
    rows = [(h, np.asarray(v, dtype=np.float16).tobytes()) for h, v in items]
    cache.executemany('INSERT OR IGNORE INTO embeddings(hash, vector) VALUES (?, ?)', rows)
    cache.commit()

@torch.inference_mode()
def encode_uncached(values):
    image_mask = np.array([is_png_b64(v) for v in values], dtype=bool)
    embeddings = np.zeros((len(values), EMBED_DIM), dtype=np.float32)

    text_idx = np.flatnonzero(~image_mask)
    for start in range(0, len(text_idx), MODEL_BATCH_SIZE):
        idx = text_idx[start:start + MODEL_BATCH_SIZE]
        inputs = processor(text=[values[i] for i in idx], padding=True, truncation=True,
                           max_length=MAX_TEXT_LENGTH, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        output = model.text_model(**inputs)
        embeddings[idx] = model.text_projection(output.pooler_output).float().cpu().numpy()

    image_idx = np.flatnonzero(image_mask)
    for start in range(0, len(image_idx), MODEL_BATCH_SIZE):
        idx = image_idx[start:start + MODEL_BATCH_SIZE]
        images, valid = [], []
        for i in idx:
            try:
                images.append(decode_png(values[i])); valid.append(i)
            except Exception as exc:
                print(f'Warning: image decode failed at local row {i}: {exc}')
        if images:
            inputs = processor(images=images, return_tensors='pt')
            pixels = inputs['pixel_values'].to(DEVICE)
            if DEVICE == 'cuda': pixels = pixels.half()
            output = model.vision_model(pixel_values=pixels)
            embeddings[np.asarray(valid)] = model.visual_projection(output.pooler_output).float().cpu().numpy()
    return l2_normalize(embeddings)

def encode_values(values):
    values = [v if isinstance(v, str) else '' for v in values]
    hashes = [object_hash(v) for v in values]
    found = cache_get_many(hashes)
    missing = {}
    for h, value in zip(hashes, values):
        if h not in found and h not in missing:
            missing[h] = value
    if missing:
        missing_hashes = list(missing)
        vectors = encode_uncached([missing[h] for h in missing_hashes])
        cache_put_many(zip(missing_hashes, vectors))
        found.update(zip(missing_hashes, vectors))
    return np.vstack([found[h] for h in hashes]), np.array(hashes), np.array([is_png_b64(v) for v in values])

## Fit caption TF-IDF models

Only captions—not Base64 images—are retained for vocabulary fitting. Duplicate captions are removed by object hash. A word model captures shared scientific terms; a character model is more robust to notation, identifiers, and small textual variations.

In [ ]:
TFIDF_PATH = WORK / 'tfidf_models.pkl'

def collect_unique_captions(csv_path):
    captions = {}
    for chunk in pd.read_csv(csv_path, usecols=['obj_1', 'obj_2'], chunksize=CSV_CHUNK_SIZE):
        for column in ['obj_1', 'obj_2']:
            for value in chunk[column].fillna(''):
                if not is_png_b64(value):
                    captions.setdefault(object_hash(value), value)
    return list(captions.values())

if TFIDF_PATH.exists():
    with open(TFIDF_PATH, 'rb') as handle:
        word_tfidf, char_tfidf = pickle.load(handle)
else:
    captions = collect_unique_captions(TRAIN_CSV)
    word_tfidf = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
                                 min_df=2, max_df=0.995, max_features=60000, dtype=np.float32)
    char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                                 max_features=60000, dtype=np.float32)
    word_tfidf.fit(captions)
    char_tfidf.fit(captions)
    with open(TFIDF_PATH, 'wb') as handle:
        pickle.dump((word_tfidf, char_tfidf), handle)
    del captions
    gc.collect()

## Perceptual hashes and pair features

The 64-bit pHash uses the low-frequency DCT of a 32×32 grayscale image. Its similarity is one minus normalized Hamming distance. TF-IDF rows are L2-normalized by `TfidfVectorizer`, so their sparse dot product is cosine similarity. Inapplicable features are zero and modality flags tell the model when to use them.

In [ ]:
def perceptual_hash(value):
    try:
        image = decode_png(value).convert('L').resize((32, 32), Image.Resampling.LANCZOS)
        coeff = dctn(np.asarray(image, dtype=np.float32), norm='ortho')[:8, :8]
        bits = coeff > np.median(coeff[1:])
        return int.from_bytes(np.packbits(bits.reshape(-1)).tobytes(), 'big')
    except Exception:
        return 0

def hamming_similarity(left, right):
    return 1.0 - ((left ^ right).bit_count() / 64.0)

def sparse_row_cosines(vectorizer, left, right, valid):
    result = np.zeros(len(left), dtype=np.float32)
    idx = np.flatnonzero(valid)
    if len(idx):
        a = vectorizer.transform([left[i] for i in idx])
        b = vectorizer.transform([right[i] for i in idx])
        result[idx] = np.asarray(a.multiply(b).sum(axis=1)).ravel()
    return result[:, None]

def make_pair_features(chunk):
    left = chunk['obj_1'].fillna('').tolist()
    right = chunk['obj_2'].fillna('').tolist()
    a, hash_a, a_img = encode_values(left)
    b, hash_b, b_img = encode_values(right)
    text_text = ~a_img & ~b_img
    image_image = a_img & b_img
    mixed = a_img ^ b_img

    cosine = np.sum(a * b, axis=1, keepdims=True)
    modalities = np.column_stack([text_text, mixed, image_image]).astype(np.float32)
    len_a = np.array([0 if flag else min(len(v), 5000) / 5000 for v, flag in zip(left, a_img)])[:, None]
    len_b = np.array([0 if flag else min(len(v), 5000) / 5000 for v, flag in zip(right, b_img)])[:, None]
    exact = np.array([valid and x.strip() == y.strip() for x, y, valid in zip(left, right, text_text)], dtype=np.float32)[:, None]

    phash_sim = np.zeros((len(left), 1), dtype=np.float32)
    for i in np.flatnonzero(image_image):
        phash_sim[i, 0] = hamming_similarity(perceptual_hash(left[i]), perceptual_hash(right[i]))
    word_sim = sparse_row_cosines(word_tfidf, left, right, text_text)
    char_sim = sparse_row_cosines(char_tfidf, left, right, text_text)

    extras = np.hstack([cosine, modalities, np.abs(len_a-len_b), np.minimum(len_a, len_b),
                        exact, phash_sim, word_sim, char_sim])
    features = np.hstack([np.abs(a-b), a*b, extras]).astype(np.float16)
    return features, hash_a, hash_b

def extract_csv(csv_path, split, has_labels):
    out_dir = WORK / split
    out_dir.mkdir(exist_ok=True)
    usecols = ['id', 'obj_1', 'obj_2'] + (LABELS if has_labels else [])
    for part, chunk in enumerate(pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE)):
        target = out_dir / f'part_{part:05d}.npz'
        if target.exists():
            continue
        X, hash_a, hash_b = make_pair_features(chunk)
        arrays = {'ids': chunk['id'].to_numpy(), 'X': X, 'hash_a': hash_a, 'hash_b': hash_b}
        if has_labels:
            arrays['y'] = chunk[LABELS].to_numpy().argmax(axis=1).astype(np.int8)
        np.savez_compressed(target, **arrays)
        if part % 25 == 0: print(split, 'part', part)
        del chunk, X, arrays
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

extract_csv(TRAIN_CSV, 'train', has_labels=True)

## Leakage-resistant groups

Pairs are edges in an object graph. Union-find assigns the same validation group to every pair connected through a repeated object hash. Consequently, an object—and its connected component—cannot occur in both the training and validation sides of a fold. This is stricter than simply grouping by one endpoint.

In [ ]:
def load_shards(split, with_labels):
    files = sorted((WORK / split).glob('part_*.npz'))
    ids, features, labels, hash_a, hash_b = [], [], [], [], []
    for path in files:
        with np.load(path) as data:
            ids.append(data['ids']); features.append(data['X'])
            hash_a.append(data['hash_a']); hash_b.append(data['hash_b'])
            if with_labels: labels.append(data['y'])
    result = [np.concatenate(ids), np.concatenate(features).astype(np.float32),
              np.concatenate(hash_a), np.concatenate(hash_b)]
    if with_labels: result.append(np.concatenate(labels))
    return tuple(result)

def connected_pair_groups(hash_a, hash_b):
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for a, b in zip(hash_a, hash_b): union(a, b)
    roots = [find(a) for a in hash_a]
    return pd.factorize(roots)[0]

train_ids, X, hash_a, hash_b, y = load_shards('train', with_labels=True)
groups = connected_pair_groups(hash_a, hash_b)
print('Rows/features/groups:', X.shape, len(np.unique(groups)))

## Out-of-fold model comparison and threshold tuning

Both candidates produce probabilities for every training row from a fold that did not train on that row. The nonlinear histogram gradient booster can learn feature interactions such as “high pHash similarity only matters for image–image pairs.” Class thresholds are coordinate-searched on OOF probabilities to maximize macro-F1. Prediction uses `argmax(p_k / t_k)`, which changes class competition without requiring probabilities to stop summing to one.

In [ ]:
def make_model(kind):
    if kind == 'linear':
        return LogisticRegression(C=2.0, class_weight='balanced', max_iter=1500, solver='lbfgs')
    if kind == 'nonlinear':
        return HistGradientBoostingClassifier(max_iter=350, learning_rate=0.06, max_leaf_nodes=31,
                                              l2_regularization=1.0, class_weight='balanced',
                                              random_state=SEED)
    raise ValueError(kind)

def oof_probabilities(kind):
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    probabilities = np.zeros((len(y), len(LABELS)), dtype=np.float32)
    scores = []
    for fold, (tr, va) in enumerate(splitter.split(X, y, groups)):
        estimator = make_model(kind)
        estimator.fit(X[tr], y[tr])
        probabilities[va] = estimator.predict_proba(X[va])
        score = f1_score(y[va], probabilities[va].argmax(axis=1), average='macro')
        scores.append(score)
        print(kind, 'fold', fold, 'macro-F1', round(score, 5))
    print(kind, 'mean macro-F1', round(float(np.mean(scores)), 5))
    return probabilities

linear_oof = oof_probabilities('linear')
nonlinear_oof = oof_probabilities('nonlinear')
linear_score = f1_score(y, linear_oof.argmax(axis=1), average='macro')
nonlinear_score = f1_score(y, nonlinear_oof.argmax(axis=1), average='macro')
BEST_KIND, best_oof = ('nonlinear', nonlinear_oof) if nonlinear_score >= linear_score else ('linear', linear_oof)
print('Selected:', BEST_KIND, 'OOF macro-F1:', max(linear_score, nonlinear_score))

def threshold_predict(probabilities, thresholds):
    return (probabilities / np.asarray(thresholds)[None, :]).argmax(axis=1)

def tune_thresholds(probabilities, targets, passes=4):
    thresholds = np.ones(probabilities.shape[1], dtype=np.float32)
    best = f1_score(targets, threshold_predict(probabilities, thresholds), average='macro')
    for _ in range(passes):
        changed = False
        for class_index in range(probabilities.shape[1]):
            local_threshold, local_score = thresholds[class_index], best
            for candidate in np.linspace(0.35, 1.8, 60):
                trial = thresholds.copy(); trial[class_index] = candidate
                score = f1_score(targets, threshold_predict(probabilities, trial), average='macro')
                if score > local_score + 1e-7:
                    local_threshold, local_score = candidate, score
            if local_score > best + 1e-7:
                thresholds[class_index], best, changed = local_threshold, local_score, True
        if not changed: break
    return thresholds, best

thresholds, tuned_score = tune_thresholds(best_oof, y)
print('Thresholds:', dict(zip(LABELS, thresholds.round(4))))
print('Tuned OOF macro-F1:', tuned_score)
print(classification_report(y, threshold_predict(best_oof, thresholds), target_names=LABELS, digits=4))

## Fit the selected model and create the submission

In [ ]:
classifier = make_model(BEST_KIND)
classifier.fit(X, y)
with open(WORK / 'classifier_and_thresholds.pkl', 'wb') as handle:
    pickle.dump({'kind': BEST_KIND, 'model': classifier, 'thresholds': thresholds}, handle)

del X, y, hash_a, hash_b, linear_oof, nonlinear_oof, best_oof
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

extract_csv(TEST_CSV, 'test', has_labels=False)
test_ids, X_test, _, _ = load_shards('test', with_labels=False)
test_probabilities = classifier.predict_proba(X_test)
pred = threshold_predict(test_probabilities, thresholds)

submission = pd.DataFrame({'id': test_ids})
for class_index, label in enumerate(LABELS):
    submission[label] = (pred == class_index).astype(np.int8)
sample = pd.read_csv(SAMPLE_CSV, usecols=lambda c: c == 'id' or c in LABELS)
submission = sample[['id']].merge(submission, on='id', how='left')
assert submission[LABELS].notna().all().all()
assert (submission[LABELS].sum(axis=1) == 1).all()
submission.to_csv('/kaggle/working/submission_improved.csv', index=False)
display(submission.head())
print('Saved submission_improved.csv:', submission.shape)

## Notes

- Save `/kaggle/working/astroclimb_improved` as a private Kaggle Dataset after extraction to reuse the cache and shards.
- If connected components leave fewer than five sufficiently populated groups, reduce `N_FOLDS`.
- Threshold optimization on the same OOF predictions used for model selection is useful for experimentation but still mildly optimistic. For rigorous comparison, nest threshold tuning inside an outer grouped cross-validation loop.
- The pHash and TF-IDF features are deliberately gated by modality. Zero means “not applicable” outside image–image and caption–caption pairs.
- For faster iteration, train the nonlinear model after the cached feature extraction has completed; no CLIP rerun is needed.